In [1]:
from time import sleep
import requests
import pandas as pd
from bs4 import BeautifulSoup
import json

In [27]:
# Helper function to scrape data and append to a list
def soup2list(src, list_, attr=None):
    if attr:
        for val in src:
            list_.append(val[attr])
    else:
        for val in src:
            list_.append(val.get_text())

# Function to scrape Trustpilot reviews for a specific company
def scrape_reviews(company, from_page=1, to_page=2):
    users = []
    ratings = []
    locations = []
    dates = []
    reviews = []

    for i in range(from_page, to_page + 1):
        print(f"Scraping page {i} for {company}...")

        # Construct the URL for the current company and page
        result = requests.get(f"https://www.trustpilot.com/review/{company}?page={i}")
        soup = BeautifulSoup(result.content, 'html.parser')

        # Scrape the relevant data
        soup2list(soup.find_all('span', {'class': 'typography_heading-xxs__QKBS8 typography_appearance-default__AAY17'}), users)
        soup2list(soup.find_all('div', {'class': 'typography_body-m__xgxZ_ typography_appearance-subtle__8_H2l styles_detailsIcon__Fo_ua'}), locations)
        soup2list(soup.find_all('div', {'class': 'styles_reviewHeader__iU9Px'}), ratings, attr='data-service-review-rating')

        dates_html = soup.find_all('div', {'class': 'styles_reviewHeader__iU9Px'})
        dates = [dates_html[i].find("time")["datetime"] for i in range(len(dates_html))]

        # Extracting titles and contents separately
        review_containers = soup.find_all('div', {'class': 'styles_reviewContent__0Q2Tg'})
        for review in review_containers:
            # First part of the review is usually the title
            title = review.find('h2').get_text() if review.find('h2') else "No Title"
            content = review.find('p').get_text() if review.find('p') else "No Content"
            review = title + " " + content
            reviews.append(review)

        sleep(5)

    # Convert the scraped data into a DataFrame
    review_data = pd.DataFrame({
        'Username': users,
        'Location': locations,
        'Date': dates,
        'Review': reviews,  # Add contents separately
        'Rating': ratings
    })

    # Return the DataFrame containing the reviews
    return review_data

# Main function to read company URLs from file and save each to a JSON file
def scrape_and_save_reviews():
    with open("urls-trustpilot.txt", 'r') as file:
        companies = [line.strip() for line in file.readlines()]  # Read and strip newlines

    for company in companies:
        print(f"Scraping reviews for {company}...")

        # Scrape reviews for the company
        review_data = scrape_reviews(company)

        # Save the scraped data to a JSON file
        json_filename = f"{company.replace('.', '_')}_reviews.json"
        review_data.to_json(json_filename, orient="records", indent=4)

        print(f"Data saved to {json_filename}")

In [2]:
result = requests.get(f"https://www.trustpilot.com/review/nordvpn.com?page=1")
soup = BeautifulSoup(result.content, 'html.parser')

In [10]:
users = soup.find_all('span', {'class': 'typography_heading-xxs__QKBS8 typography_appearance-default__AAY17'})
locations = soup.find_all('div', {'class': 'typography_body-m__xgxZ_ typography_appearance-subtle__8_H2l styles_detailsIcon__Fo_ua'})
ratings = soup.find_all('div', {'class': 'styles_reviewHeader__iU9Px'})

In [13]:
ratings[0]['data-service-review-rating']

'5'

In [29]:
reviews_nordvpn = scrape_reviews(company="nordvpn.com", from_page=1, to_page=1)

Scraping page 1 for nordvpn.com...


In [30]:
reviews_nordvpn

,Username,Location,Date,Content,Rating
0,"Sele, Birmingham",GB,2024-10-06T13:12:36.000Z,Amazing support! Since my retirement I've subs...,5
1,Mr JOHN ROBERT TAYLOR,GB,2024-10-05T11:07:27.000Z,The Best on the Market Over the years I’ve be...,5
2,sf,US,2024-10-06T18:30:45.000Z,Absolutely GARBAGE WORTHLESS service… Absolute...,3
3,D Moon,KR,2024-10-05T16:02:41.000Z,I tried to make a refund I tried to make a ref...,5
4,Christopher Hegan,GB,2024-10-05T01:53:56.000Z,"Reliable and fast connections, security, vigil...",5
5,Silvana nikolle Guerrero padil,ES,2024-10-04T00:14:54.000Z,Trustpilot is here to help you shape… Trustpil...,5
6,Truthizm -,AU,2024-10-04T10:25:37.000Z,Fast and Easy to set up. Bernanda helped me s...,5
7,Andrew,AU,2024-10-04T09:29:52.000Z,Persevere with Nord VPN - it's worth it. I fin...,4
8,murray shaw,GB,2024-10-02T20:05:16.000Z,Nord assistance Got Nord 12 months ago to prov...,5
9,Brian,DE,2024-10-06T11:31:57.000Z,Super Customer Service and Troubleshooting I a...,5
